# Mission Partie 1 — HR Attrition

Objectif des 3 premières étapes :

1. Préparer l'environnement projet.
2. Charger, comprendre, nettoyer et joindre les 3 fichiers.
3. Construire `X` et `y` prêts pour une future modélisation sklearn.

## Étape 1 — Imports et chargement des fichiers

Place les 3 CSV dans un dossier `data/` à la racine du projet.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# missingno sert seulement à visualiser les valeurs manquantes.
# On le rend optionnel pour éviter que le notebook plante si le package n'est pas installé.
try:
    import missingno as msno
except ImportError:
    msno = None
    print("Package missingno non installé : ce n'est pas bloquant pour la suite du projet.")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Le notebook est dans le dossier notebooks/.
# Les données sont dans ../data si on lance depuis notebooks/.
# Si on lance depuis la racine du projet, on vérifie aussi data/.
possible_data_dirs = [Path("../data"), Path("data"), Path("/mnt/data/work/data")]
DATA_DIR = next((p for p in possible_data_dirs if p.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError("Impossible de trouver le dossier data. Placez les CSV dans un dossier data/.")

sirh_path = DATA_DIR / "extrait_sirh.csv"
eval_path = DATA_DIR / "extrait_eval.csv"
sondage_path = DATA_DIR / "extrait_sondage.csv"

sirh = pd.read_csv(sirh_path)
evaluations = pd.read_csv(eval_path)
sondage = pd.read_csv(sondage_path)

print("Dossier de données utilisé :", DATA_DIR.resolve())
print("SIRH :", sirh.shape)
print("Évaluations :", evaluations.shape)
print("Sondage :", sondage.shape)

## Étape 2 — Compréhension rapide des fichiers

In [ ]:
def audit_dataframe(df: pd.DataFrame, name: str) -> None:
    print(f"\n===== {name} =====")
    display(df.head())
    display(pd.DataFrame({
        "dtype": df.dtypes,
        "nb_valeurs_manquantes": df.isna().sum(),
        "taux_valeurs_manquantes": (df.isna().mean() * 100).round(2),
        "nb_modalites": df.nunique(dropna=False)
    }))
    print("Doublons :", df.duplicated().sum())

audit_dataframe(sirh, "SIRH")
audit_dataframe(evaluations, "Évaluations")
audit_dataframe(sondage, "Sondage")

### Nettoyage simple des colonnes

On corrige surtout :
- `eval_number` pour récupérer l'identifiant numérique de l'employé ;
- `augementation_salaire_precedente` pour obtenir un nombre ;
- quelques variables Oui/Non ou Y en variables binaires.

In [ ]:
def clean_percentage(value):
    """Transforme une valeur comme '11 %' en 11.0."""
    if pd.isna(value):
        return np.nan
    return float(str(value).replace("%", "").strip())

sirh_clean = sirh.copy()
eval_clean = evaluations.copy()
sondage_clean = sondage.copy()

# Clé de jointure extraite depuis E_1, E_2, etc.
eval_clean["id_employee"] = (
    eval_clean["eval_number"]
    .astype(str)
    .str.extract(r"(\d+)")
    .astype(int)
)

# Pourcentage d'augmentation salariale
eval_clean["augmentation_salaire_precedente_pct"] = (
    eval_clean["augementation_salaire_precedente"]
    .apply(clean_percentage)
)

# Variables binaires
eval_clean["heure_supplementaires"] = eval_clean["heure_supplementaires"].map({"Oui": 1, "Non": 0})
sondage_clean["a_quitte_l_entreprise"] = sondage_clean["a_quitte_l_entreprise"].map({"Oui": 1, "Non": 0})
sondage_clean["ayant_enfants"] = sondage_clean["ayant_enfants"].map({"Y": 1, "N": 0})

# Renommer la clé sondage
sondage_clean = sondage_clean.rename(columns={"code_sondage": "id_employee"})

display(eval_clean.head())
display(sondage_clean.head())

### Jointure des 3 fichiers

Les 3 fichiers contiennent les mêmes employés. On peut donc utiliser une jointure `inner` pour garder uniquement les employés présents dans les 3 sources.

In [ ]:
df = (
    sirh_clean
    .merge(sondage_clean, on="id_employee", how="inner")
    .merge(eval_clean, on="id_employee", how="inner")
)

print("Dimensions du DataFrame central :", df.shape)
display(df.head())

# Contrôle de la cible
display(df["a_quitte_l_entreprise"].value_counts())
display((df["a_quitte_l_entreprise"].value_counts(normalize=True) * 100).round(2))

### Statistiques descriptives pour comparer les démissionnaires et les employés encore présents

In [ ]:
target_col = "a_quitte_l_entreprise"

num_cols = df.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]

stats_quanti = df.groupby(target_col)[num_cols].mean().T
stats_quanti.columns = ["moyenne_non_quitte", "moyenne_quitte"]
stats_quanti["ecart_quitte_moins_non"] = stats_quanti["moyenne_quitte"] - stats_quanti["moyenne_non_quitte"]
stats_quanti["ecart_absolu"] = stats_quanti["ecart_quitte_moins_non"].abs()

display(stats_quanti.sort_values("ecart_absolu", ascending=False).head(20))

In [ ]:
cat_cols = df.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    if col in ["eval_number", "augementation_salaire_precedente"]:
        continue

    table = (
        df.groupby(col)[target_col]
        .agg(taux_attrition="mean", effectif="count")
        .sort_values("taux_attrition", ascending=False)
    )
    table["taux_attrition"] = (table["taux_attrition"] * 100).round(1)
    print(f"\nVariable : {col}")
    display(table)

### Visualisations EDA

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=df, x=target_col)
plt.title("Répartition de la cible : a quitté l'entreprise")
plt.xlabel("A quitté l'entreprise")
plt.ylabel("Nombre d'employés")
plt.show()

In [ ]:
variables_quanti_interessantes = [
    "age",
    "revenu_mensuel",
    "annee_experience_totale",
    "annees_dans_l_entreprise",
    "distance_domicile_travail",
    "annees_dans_le_poste_actuel",
    "satisfaction_employee_environnement",
    "satisfaction_employee_equilibre_pro_perso",
]

for col in variables_quanti_interessantes:
    if col in df.columns:
        plt.figure(figsize=(7, 4))
        sns.boxplot(data=df, x=target_col, y=col)
        plt.title(f"{col} selon l'attrition")
        plt.xlabel("A quitté l'entreprise")
        plt.ylabel(col)
        plt.show()

In [ ]:
variables_quali_interessantes = [
    "poste",
    "departement",
    "statut_marital",
    "frequence_deplacement",
    "heure_supplementaires",
    "domaine_etude",
]

for col in variables_quali_interessantes:
    if col in df.columns:
        taux = (
            df.groupby(col)[target_col]
            .mean()
            .sort_values(ascending=False)
            .mul(100)
        )
        plt.figure(figsize=(9, 4))
        sns.barplot(x=taux.index, y=taux.values)
        plt.title(f"Taux d'attrition par {col}")
        plt.ylabel("Taux d'attrition (%)")
        plt.xlabel(col)
        plt.xticks(rotation=45, ha="right")
        plt.show()

## Étape 3 — Préparation de `X` et `y` pour sklearn

Objectif : obtenir :
- `y` : la cible `a_quitte_l_entreprise` ;
- `X` : un DataFrame numérique, sans identifiants inutiles, sans colonne cible, sans variable constante, et encodé.

In [ ]:
def build_modeling_dataset(df: pd.DataFrame):
    df_model = df.copy()

    target = "a_quitte_l_entreprise"

    # Colonnes à supprimer : cible, identifiants, colonnes redondantes ou brutes après nettoyage
    cols_to_drop = [
        target,
        "id_employee",
        "eval_number",
        "augementation_salaire_precedente",
    ]
    cols_to_drop = [c for c in cols_to_drop if c in df_model.columns]

    y = df_model[target].astype(int)
    X = df_model.drop(columns=cols_to_drop)

    # Supprimer les variables constantes, car elles n'apportent aucune information au modèle
    constant_cols = [col for col in X.columns if X[col].nunique(dropna=False) <= 1]
    X = X.drop(columns=constant_cols)
    print("Colonnes constantes supprimées :", constant_cols)

    # Séparer colonnes quantitatives et qualitatives
    numeric_features = X.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

    # Encodage OneHot des variables qualitatives nominales
    X_encoded = pd.get_dummies(
        X,
        columns=categorical_features,
        drop_first=True,
        dtype=int
    )

    # Sécurité : remplacer d'éventuelles valeurs manquantes restantes
    X_encoded = X_encoded.fillna(X_encoded.median(numeric_only=True))

    return X_encoded, y

X, y = build_modeling_dataset(df)

print("X :", X.shape)
print("y :", y.shape)
display(X.head())
display(y.head())

### Analyse des corrélations entre variables numériques

In [ ]:
corr = X.corr(method="pearson")

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Matrice de corrélation de Pearson sur X encodé")
plt.show()

# Repérer les corrélations très fortes
threshold = 0.90
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "correlation"})
)
high_corr_pairs = high_corr_pairs[high_corr_pairs["correlation"].abs() >= threshold]
display(high_corr_pairs.sort_values("correlation", key=lambda s: s.abs(), ascending=False))

In [ ]:
# Option : supprimer automatiquement une des deux variables si corrélation > 0.90
features_to_drop_corr = [
    column for column in upper.columns 
    if any(upper[column].abs() > threshold)
]

print("Features supprimables car trop corrélées :", features_to_drop_corr)

X_reduced = X.drop(columns=features_to_drop_corr)

print("X réduit :", X_reduced.shape)
display(X_reduced.head())

## Premières observations à présenter à Amandine

À compléter avec tes graphiques, mais les premiers résultats à vérifier sont :

- Le taux d'attrition global est d'environ 16 %.
- Les employés ayant quitté l'entreprise semblent avoir en moyenne un revenu mensuel plus bas.
- Ils semblent aussi être plus jeunes, avec moins d'expérience totale et moins d'ancienneté dans l'entreprise.
- Les heures supplémentaires sont associées à un taux d'attrition plus élevé.
- Les déplacements fréquents semblent aussi associés à un taux d'attrition plus élevé.
- Certains postes commerciaux ou de consultant semblent plus exposés que les postes plus seniors ou techniques.

Attention : ces constats sont descriptifs. Ils ne prouvent pas encore une causalité. La modélisation et SHAP permettront ensuite d'identifier les variables les plus explicatives.

# Étape 4 — Modélisation supervisée

Dans cette partie, on termine le travail demandé dans l'énoncé.

Objectif : entraîner plusieurs modèles pour prédire si un salarié risque de quitter l'entreprise.

On va construire :

- un modèle **Dummy**, qui sert de référence minimale ;
- un modèle **linéaire**, ici une régression logistique ;
- un modèle **non linéaire**, ici un Random Forest.

Comme la classe `a_quitte_l_entreprise = 1` est minoritaire, on utilise une séparation **stratifiée** et des modèles capables de tenir compte du déséquilibre des classes.

## 4.1 — Ajout de features métier

Avant de lancer les modèles, on crée quelques variables supplémentaires à partir des colonnes existantes.

Ces nouvelles variables ne sont pas inventées au hasard : elles traduisent des idées métier utiles pour les RH.

Exemples :

- une satisfaction moyenne ;
- l'écart entre l'évaluation actuelle et précédente ;
- un indicateur de retard de promotion ;
- un ratio d'ancienneté dans le poste ;
- un indicateur simple de charge liée aux heures supplémentaires et aux déplacements fréquents.

In [ ]:
def add_business_features(df: pd.DataFrame) -> pd.DataFrame:
    """Ajoute des variables métier simples pour enrichir la modélisation.

    Le but est d'améliorer la capacité du modèle à détecter les profils à risque,
    tout en gardant des variables compréhensibles pour une équipe RH.
    """
    df_fe = df.copy()

    satisfaction_cols = [
        "satisfaction_employee_environnement",
        "satisfaction_employee_nature_travail",
        "satisfaction_employee_equipe",
        "satisfaction_employee_equilibre_pro_perso",
    ]

    # Satisfaction moyenne globale du salarié.
    df_fe["satisfaction_moyenne"] = df_fe[satisfaction_cols].mean(axis=1)

    # Évolution de la note d'évaluation entre l'année précédente et l'année actuelle.
    df_fe["ecart_note_evaluation"] = (
        df_fe["note_evaluation_actuelle"] - df_fe["note_evaluation_precedente"]
    )

    # Part de l'ancienneté passée dans le poste actuel.
    df_fe["ratio_anciennete_poste"] = (
        df_fe["annees_dans_le_poste_actuel"] / (df_fe["annees_dans_l_entreprise"] + 1)
    )

    # Part de l'ancienneté passée avec le responsable actuel.
    df_fe["ratio_anciennete_responsable"] = (
        df_fe["annes_sous_responsable_actuel"] / (df_fe["annees_dans_l_entreprise"] + 1)
    )

    # Expérience acquise avant l'arrivée dans l'entreprise.
    df_fe["experience_avant_entreprise"] = (
        df_fe["annee_experience_totale"] - df_fe["annees_dans_l_entreprise"]
    ).clip(lower=0)

    # Salaire rapporté à l'expérience totale.
    df_fe["salaire_par_annee_experience"] = (
        df_fe["revenu_mensuel"] / (df_fe["annee_experience_totale"] + 1)
    )

    # Indicateur de promotion ancienne : 1 si la dernière promotion date d'au moins 5 ans.
    df_fe["promotion_ancienne"] = (
        df_fe["annees_depuis_la_derniere_promotion"] >= 5
    ).astype(int)

    # Indicateur de charge : heures supplémentaires + déplacements fréquents.
    df_fe["charge_operationnelle_risque"] = (
        df_fe["heure_supplementaires"].astype(int)
        + (df_fe["frequence_deplacement"] == "Frequent").astype(int)
    )

    return df_fe


df_fe = add_business_features(df)

X_fe, y = build_modeling_dataset(df_fe)

# On retire de nouveau les variables très corrélées, mais cette fois sur le dataset enrichi.
corr_fe = X_fe.corr(method="pearson")
upper_fe = corr_fe.where(np.triu(np.ones(corr_fe.shape), k=1).astype(bool))
threshold = 0.90
features_to_drop_corr_fe = [
    column for column in upper_fe.columns
    if any(upper_fe[column].abs() > threshold)
]

X_final = X_fe.drop(columns=features_to_drop_corr_fe)

print("Nombre de features avant suppression des corrélations :", X_fe.shape[1])
print("Features supprimées car trop corrélées :", features_to_drop_corr_fe)
print("Nombre de features finales :", X_final.shape[1])

display(X_final.head())

**Petite conclusion :**

On a maintenant un jeu de données enrichi, entièrement numérique, prêt pour l'entraînement des modèles.  
Les nouvelles variables permettent de mieux traduire des réalités RH : satisfaction, évolution de performance, charge de travail, promotion et ancienneté.

## 4.2 — Séparation train/test stratifiée

On sépare les données en deux parties :

- `X_train`, `y_train` : pour entraîner les modèles ;
- `X_test`, `y_test` : pour vérifier si les modèles généralisent bien sur des données jamais vues.

On utilise `stratify=y` car la classe des salariés ayant quitté l'entreprise est minoritaire.  
La stratification permet de garder à peu près le même taux d'attrition dans le train et dans le test.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensions X_train :", X_train.shape)
print("Dimensions X_test  :", X_test.shape)
print("Dimensions y_train :", y_train.shape)
print("Dimensions y_test  :", y_test.shape)

repartition = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
})
repartition.index = ["Reste dans l'entreprise", "A quitté l'entreprise"]

display(repartition.style.format("{:.2%}"))

**Petite conclusion :**

La séparation est correcte si le taux d'attrition est similaire dans le jeu d'entraînement et dans le jeu de test.  
Cela évite d'entraîner ou d'évaluer le modèle sur une répartition trop différente de la réalité.

## 4.3 — Fonction d'évaluation des modèles

Pour comparer proprement les modèles, on crée une fonction qui calcule les mêmes métriques à chaque fois.

Les métriques importantes sont :

- **accuracy** : part totale de bonnes prédictions ;
- **precision** : parmi les salariés prédits comme partants, combien sont vraiment partis ;
- **recall** : parmi les salariés vraiment partis, combien le modèle détecte ;
- **f1-score** : compromis entre précision et rappel ;
- **ROC-AUC** : capacité globale à séparer les deux classes ;
- **Average Precision** : performance sur la courbe précision-rappel, utile en cas de classes déséquilibrées.

Dans notre contexte RH, le **recall** est particulièrement important, car on veut éviter de manquer trop de salariés réellement à risque.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
)


def get_positive_scores(model, X_data):
    """Récupère un score de probabilité pour la classe positive quand c'est possible."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X_data)
    return None


def evaluate_model(model, X_data, y_true, dataset_name: str, model_name: str, threshold: float = 0.5):
    """Calcule les métriques principales pour un modèle donné."""
    scores = get_positive_scores(model, X_data)

    if scores is not None:
        y_pred = (scores >= threshold).astype(int)
    else:
        y_pred = model.predict(X_data)

    results = {
        "modele": model_name,
        "dataset": dataset_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

    if scores is not None and len(np.unique(y_true)) == 2:
        results["roc_auc"] = roc_auc_score(y_true, scores)
        results["average_precision"] = average_precision_score(y_true, scores)
    else:
        results["roc_auc"] = np.nan
        results["average_precision"] = np.nan

    return results


def print_detailed_evaluation(model, X_data, y_true, model_name: str, dataset_name: str, threshold: float = 0.5):
    """Affiche le rapport de classification et la matrice de confusion."""
    scores = get_positive_scores(model, X_data)
    if scores is not None:
        y_pred = (scores >= threshold).astype(int)
    else:
        y_pred = model.predict(X_data)

    print(f"===== {model_name} — {dataset_name} — seuil {threshold:.2f} =====")
    print(classification_report(y_true, y_pred, target_names=["Reste", "Quitte"], zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Reste", "Quitte"])
    disp.plot(values_format="d")
    plt.title(f"Matrice de confusion — {model_name} — {dataset_name}")
    plt.show()

**Petite conclusion :**

Cette fonction évite de recopier le même code pour chaque modèle.  
Elle nous permettra aussi de vérifier l'overfitting en comparant les résultats sur le train et sur le test.

## 4.4 — Modèle Dummy

Le modèle Dummy est un modèle étalon.

Il sert à répondre à une question simple :

> Est-ce que nos vrais modèles font mieux qu'un modèle très basique ?

Ici, on utilise la stratégie `most_frequent`, donc le modèle prédit toujours la classe majoritaire : les salariés qui restent.

In [ ]:
from sklearn.dummy import DummyClassifier

models = {}
all_results = []

dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_model.fit(X_train, y_train)
models["Dummy"] = dummy_model

for dataset_name, X_data, y_data in [
    ("train", X_train, y_train),
    ("test", X_test, y_test),
]:
    all_results.append(evaluate_model(dummy_model, X_data, y_data, dataset_name, "Dummy"))

print_detailed_evaluation(dummy_model, X_test, y_test, "Dummy", "test")

results_df = pd.DataFrame(all_results)
display(results_df.style.format({
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}",
}))

**Petite conclusion :**

Le Dummy peut avoir une accuracy correcte, car la majorité des salariés restent.  
Mais il ne détecte pas les salariés qui quittent l'entreprise.  
C'est donc un modèle inutile pour les RH, mais indispensable comme point de comparaison.

## 4.5 — Modèle linéaire : Régression logistique

La régression logistique est un modèle linéaire classique pour la classification binaire.

On l'utilise avec :

- `StandardScaler`, pour mettre les variables sur une échelle comparable ;
- `class_weight="balanced"`, pour tenir compte du déséquilibre entre salariés restés et salariés partis.

Ce modèle est souvent moins puissant qu'un modèle à base d'arbres, mais il est plus simple à expliquer.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=42,
    )),
])

logistic_model.fit(X_train, y_train)
models["Régression logistique"] = logistic_model

for dataset_name, X_data, y_data in [
    ("train", X_train, y_train),
    ("test", X_test, y_test),
]:
    all_results.append(evaluate_model(logistic_model, X_data, y_data, dataset_name, "Régression logistique"))

print_detailed_evaluation(logistic_model, X_test, y_test, "Régression logistique", "test")

results_df = pd.DataFrame(all_results)
display(results_df.style.format({
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}",
}))

**Petite conclusion :**

La régression logistique sert à vérifier si une relation relativement simple suffit à détecter les départs.  
Si elle obtient déjà un bon rappel, cela signifie que les signaux d'attrition sont assez visibles dans les données.

## 4.6 — Modèle non linéaire : Random Forest

Le Random Forest est un modèle à base d'arbres.

Il est capable de détecter des relations plus complexes entre les variables.

Exemple : le risque de départ peut dépendre en même temps du salaire, de l'ancienneté, du poste, des heures supplémentaires et de la fréquence de déplacement.

On utilise aussi `class_weight="balanced"` pour tenir compte du déséquilibre des classes.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=1,
)

rf_model.fit(X_train, y_train)
models["Random Forest"] = rf_model

for dataset_name, X_data, y_data in [
    ("train", X_train, y_train),
    ("test", X_test, y_test),
]:
    all_results.append(evaluate_model(rf_model, X_data, y_data, dataset_name, "Random Forest"))

print_detailed_evaluation(rf_model, X_test, y_test, "Random Forest", "test")

results_df = pd.DataFrame(all_results)
display(results_df.style.format({
    "accuracy": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "roc_auc": "{:.3f}",
    "average_precision": "{:.3f}",
}))

**Petite conclusion :**

Le Random Forest est le premier modèle non linéaire du projet.  
Il permet de mieux capter les interactions entre variables.  
La comparaison train/test permet de voir s'il apprend réellement ou s'il sur-apprend les données d'entraînement.

## 4.7 — Comparaison synthétique des modèles

On rassemble toutes les métriques dans un tableau pour comparer les modèles.

In [ ]:
results_df = pd.DataFrame(all_results)

metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
comparison = results_df.sort_values(["dataset", "f1"], ascending=[True, False])

display(comparison.style.format({col: "{:.3f}" for col in metric_cols}))

plt.figure(figsize=(10, 5))
sns.barplot(
    data=results_df[results_df["dataset"] == "test"],
    x="modele",
    y="recall"
)
plt.title("Comparaison du recall sur le jeu de test")
plt.ylabel("Recall classe attrition")
plt.xlabel("Modèle")
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(
    data=results_df[results_df["dataset"] == "test"],
    x="modele",
    y="f1"
)
plt.title("Comparaison du F1-score sur le jeu de test")
plt.ylabel("F1-score classe attrition")
plt.xlabel("Modèle")
plt.xticks(rotation=20)
plt.show()

**Petite conclusion :**

Le meilleur modèle n'est pas forcément celui avec la meilleure accuracy.  
Dans ce projet RH, on cherche surtout un bon compromis entre recall et precision.  
Le recall est important pour identifier le plus possible de salariés réellement à risque.

# Étape 5 — Validation croisée et prise en compte du contexte métier

L'évaluation sur un seul train/test split peut dépendre du découpage choisi.  
Pour sécuriser l'analyse, on ajoute une validation croisée stratifiée.

Cela permet d'obtenir :

- une moyenne des performances ;
- un écart-type ;
- une meilleure idée de la stabilité du modèle.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
}

cv_rows = []

for model_name, model in models.items():
    cv_scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=1,
    )

    for metric in scoring.keys():
        cv_rows.append({
            "modele": model_name,
            "metrique": metric,
            "train_moyenne": cv_scores[f"train_{metric}"].mean(),
            "train_ecart_type": cv_scores[f"train_{metric}"].std(),
            "validation_moyenne": cv_scores[f"test_{metric}"].mean(),
            "validation_ecart_type": cv_scores[f"test_{metric}"].std(),
        })

cv_results_df = pd.DataFrame(cv_rows)

display(cv_results_df.style.format({
    "train_moyenne": "{:.3f}",
    "train_ecart_type": "{:.3f}",
    "validation_moyenne": "{:.3f}",
    "validation_ecart_type": "{:.3f}",
}))

**Petite conclusion :**

La validation croisée donne une vision plus fiable que le simple jeu de test.  
Si les scores train sont beaucoup plus élevés que les scores validation, cela indique un risque d'overfitting.  
Si l'écart-type est élevé, cela signifie que le modèle est instable selon les folds.

## 5.1 — Courbe précision-rappel et choix du seuil

Par défaut, un modèle classe un salarié comme partant si la probabilité prédite est supérieure à 0,50.

Mais dans un contexte RH, on peut vouloir modifier ce seuil.

Exemple :

- seuil plus bas : on détecte plus de salariés à risque, mais on augmente les faux positifs ;
- seuil plus haut : on alerte moins souvent, mais on risque de manquer des salariés qui vont vraiment partir.

On utilise donc la courbe précision-rappel pour choisir un seuil plus adapté au besoin métier.

In [ ]:
from sklearn.metrics import precision_recall_curve

# On choisit le Random Forest comme modèle de référence avant fine-tuning,
# car il est non linéaire et compatible avec l'interprétation par SHAP TreeExplainer.
base_model = rf_model
base_model_name = "Random Forest"

y_test_scores = get_positive_scores(base_model, X_test)
precisions, recalls, thresholds = precision_recall_curve(y_test, y_test_scores)

# Calcul du F1 pour chaque seuil. La dernière valeur precision/recall n'a pas de seuil associé.
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-12)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Meilleur seuil selon le F1-score :", round(best_threshold, 3))
print("Precision à ce seuil :", round(precisions[best_idx], 3))
print("Recall à ce seuil :", round(recalls[best_idx], 3))
print("F1-score à ce seuil :", round(f1_scores[best_idx], 3))

plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions, marker=".")
plt.title("Courbe précision-rappel — Random Forest")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()

print_detailed_evaluation(base_model, X_test, y_test, base_model_name, "test", threshold=best_threshold)

**Petite conclusion :**

Le choix du seuil est une décision métier.  
Pour les RH, un seuil plus bas peut être utile si l'objectif est de détecter plus largement les salariés à risque et de déclencher un entretien préventif.  
Il faudra cependant accepter plus de faux positifs.

# Étape 6 — Fine-tuning du modèle non linéaire

On améliore maintenant le Random Forest avec une recherche d'hyperparamètres.

Le but est de trouver une meilleure combinaison de paramètres, par exemple :

- le nombre d'arbres ;
- la profondeur maximale ;
- le nombre minimum d'observations par feuille ;
- le nombre de variables testées à chaque séparation.

On optimise ici le `average_precision`, car cette métrique est adaptée aux classes déséquilibrées.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_distributions = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 4, 6, 8, 12],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.5],
}

rf_for_search = RandomForestClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=1,
)

search = RandomizedSearchCV(
    estimator=rf_for_search,
    param_distributions=param_distributions,
    n_iter=8,
    scoring="average_precision",
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1,
)

search.fit(X_train, y_train)

best_rf = search.best_estimator_
models["Random Forest optimisé"] = best_rf

print("Meilleurs paramètres :")
display(search.best_params_)
print("Meilleur score CV average_precision :", round(search.best_score_, 3))

for dataset_name, X_data, y_data in [
    ("train", X_train, y_train),
    ("test", X_test, y_test),
]:
    all_results.append(evaluate_model(best_rf, X_data, y_data, dataset_name, "Random Forest optimisé"))

print_detailed_evaluation(best_rf, X_test, y_test, "Random Forest optimisé", "test")

results_df = pd.DataFrame(all_results)
display(results_df[results_df["modele"].isin(["Random Forest", "Random Forest optimisé"])]
        .style.format({col: "{:.3f}" for col in metric_cols}))

**Petite conclusion :**

Le fine-tuning permet d'améliorer ou de stabiliser le modèle.  
Il ne faut pas seulement regarder si le score augmente : il faut aussi vérifier que l'écart entre train et test reste raisonnable.  
Un modèle très fort sur le train mais faible sur le test serait en overfitting.

## 6.1 — Nouveau choix de seuil pour le modèle optimisé

Après optimisation, on recalcule un seuil adapté à partir de la courbe précision-rappel.

In [ ]:
best_scores = get_positive_scores(best_rf, X_test)
precisions_opt, recalls_opt, thresholds_opt = precision_recall_curve(y_test, best_scores)
f1_scores_opt = 2 * (precisions_opt[:-1] * recalls_opt[:-1]) / (precisions_opt[:-1] + recalls_opt[:-1] + 1e-12)
best_idx_opt = np.argmax(f1_scores_opt)
best_threshold_opt = thresholds_opt[best_idx_opt]

print("Meilleur seuil optimisé selon le F1-score :", round(best_threshold_opt, 3))
print("Precision à ce seuil :", round(precisions_opt[best_idx_opt], 3))
print("Recall à ce seuil :", round(recalls_opt[best_idx_opt], 3))
print("F1-score à ce seuil :", round(f1_scores_opt[best_idx_opt], 3))

plt.figure(figsize=(7, 5))
plt.plot(recalls_opt, precisions_opt, marker=".")
plt.title("Courbe précision-rappel — Random Forest optimisé")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.grid(True)
plt.show()

print_detailed_evaluation(best_rf, X_test, y_test, "Random Forest optimisé", "test", threshold=best_threshold_opt)

**Petite conclusion :**

Le seuil final doit être choisi avec les RH.  
Si l'objectif est de ne pas rater les salariés à risque, on privilégiera un meilleur recall.  
Si l'objectif est d'éviter trop d'alertes inutiles, on privilégiera davantage la précision.

# Étape 7 — Interprétation du modèle : feature importance et SHAP

Une fois le modèle optimisé, on doit expliquer ses décisions.

On va comparer trois approches :

- l'importance native du Random Forest ;
- la permutation importance de sklearn ;
- les valeurs SHAP.

L'objectif est de transformer le modèle en explications compréhensibles pour les RH.

## 7.1 — Importance native du Random Forest

In [ ]:
native_importance = pd.DataFrame({
    "feature": X_final.columns,
    "importance_native_rf": best_rf.feature_importances_,
}).sort_values("importance_native_rf", ascending=False)

display(native_importance.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=native_importance.head(15),
    x="importance_native_rf",
    y="feature"
)
plt.title("Top 15 des variables — Importance native Random Forest")
plt.xlabel("Importance")
plt.ylabel("Variable")
plt.show()

**Petite conclusion :**

Cette première méthode donne une vision rapide des variables les plus utilisées par le Random Forest.  
Elle est utile, mais elle peut être biaisée. C'est pour cela qu'on compare avec d'autres méthodes.

## 7.2 — Permutation Importance

La permutation importance mesure la baisse de performance quand on mélange les valeurs d'une variable.

Si mélanger une variable dégrade fortement le modèle, cela signifie que cette variable est importante.

In [ ]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    best_rf,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="average_precision",
    n_jobs=1,
)

perm_importance = pd.DataFrame({
    "feature": X_final.columns,
    "importance_moyenne": perm_result.importances_mean,
    "importance_ecart_type": perm_result.importances_std,
}).sort_values("importance_moyenne", ascending=False)

display(perm_importance.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(
    data=perm_importance.head(15),
    x="importance_moyenne",
    y="feature"
)
plt.title("Top 15 des variables — Permutation Importance")
plt.xlabel("Baisse moyenne de performance")
plt.ylabel("Variable")
plt.show()

**Petite conclusion :**

La permutation importance est intéressante parce qu'elle évalue l'impact réel d'une variable sur la performance du modèle.  
Les variables qui ressortent ici sont de bons candidats pour expliquer les causes potentielles d'attrition.

## 7.3 — SHAP : explication globale

SHAP permet de comprendre l'influence des variables sur les prédictions.

Une valeur SHAP positive pousse la prédiction vers la classe `1`, donc vers le départ du salarié.  
Une valeur SHAP négative pousse la prédiction vers la classe `0`, donc vers le maintien dans l'entreprise.

In [ ]:
import shap

# Pour éviter des temps de calcul trop longs, on peut expliquer un échantillon du test.
X_test_sample = X_test.sample(n=min(120, len(X_test)), random_state=42)

explainer = shap.TreeExplainer(best_rf)
shap_values = explainer.shap_values(X_test_sample)

# Selon la version de shap, le format peut être une liste ou un tableau 3D.
if isinstance(shap_values, list):
    shap_values_positive = shap_values[1]
elif getattr(shap_values, "ndim", None) == 3:
    shap_values_positive = shap_values[:, :, 1]
else:
    shap_values_positive = shap_values

shap.summary_plot(shap_values_positive, X_test_sample, plot_type="bar", show=True)
shap.summary_plot(shap_values_positive, X_test_sample, show=True)

**Petite conclusion :**

Le graphique SHAP global permet d'identifier les variables qui influencent le plus le risque de départ.  
Le beeswarm plot permet aussi de voir dans quel sens les valeurs jouent : par exemple, une valeur élevée peut augmenter ou diminuer le risque selon la variable.

## 7.4 — SHAP : exemples locaux avec Waterfall Plot

La feature importance globale explique le modèle dans son ensemble.

La feature importance locale explique une prédiction individuelle.

On choisit ici :

- un exemple de salarié ayant quitté l'entreprise ;
- un exemple de salarié resté dans l'entreprise.

In [ ]:
# On prépare des objets Explanation pour faire des Waterfall Plots.
expected_value = explainer.expected_value
if isinstance(expected_value, (list, np.ndarray)):
    expected_value_positive = expected_value[1] if len(expected_value) > 1 else expected_value[0]
else:
    expected_value_positive = expected_value

# Sélection d'un salarié réellement parti et d'un salarié réellement resté dans l'échantillon.
indices_quitte = y_test.loc[X_test_sample.index][y_test.loc[X_test_sample.index] == 1].index
indices_reste = y_test.loc[X_test_sample.index][y_test.loc[X_test_sample.index] == 0].index

selected_indices = []
if len(indices_quitte) > 0:
    selected_indices.append(indices_quitte[0])
if len(indices_reste) > 0:
    selected_indices.append(indices_reste[0])

for idx in selected_indices:
    row_position = list(X_test_sample.index).index(idx)
    vraie_classe = y_test.loc[idx]
    proba_depart = best_rf.predict_proba(X_test_sample.loc[[idx]])[:, 1][0]

    print("=" * 80)
    print(f"Employé index {idx}")
    print("Classe réelle :", "A quitté" if vraie_classe == 1 else "Est resté")
    print("Probabilité prédite de départ :", round(proba_depart, 3))

    explanation = shap.Explanation(
        values=shap_values_positive[row_position],
        base_values=expected_value_positive,
        data=X_test_sample.iloc[row_position],
        feature_names=X_test_sample.columns,
    )

    shap.plots.waterfall(explanation, max_display=12)

**Petite conclusion :**

Le Waterfall Plot permet d'expliquer une prédiction salarié par salarié.  
C'est très utile pour un usage RH, car on peut comprendre pourquoi un profil est considéré comme plus ou moins à risque.

# Étape 8 — Synthèse finale pour la présentation

Cette dernière partie sert à préparer les messages à présenter à Amandine et aux RH.

In [ ]:
# Tableau final de comparaison des modèles sur le jeu de test.
final_results = pd.DataFrame(all_results)
final_test_results = final_results[final_results["dataset"] == "test"].sort_values("average_precision", ascending=False)

display(final_test_results.style.format({col: "{:.3f}" for col in metric_cols}))

print("Modèle retenu : Random Forest optimisé")
print("Seuil recommandé à discuter avec les RH :", round(best_threshold_opt, 3))
print("\nTop 10 variables selon SHAP / importance native / permutation à analyser dans la présentation :")

top_features = (
    native_importance.head(10)[["feature"]]
    .merge(perm_importance[["feature", "importance_moyenne"]], on="feature", how="left")
)
display(top_features)

## Conclusion générale à mettre dans le rapport

L'analyse exploratoire montre que l'attrition concerne une minorité importante de salariés.  
Les premiers signaux observés concernent notamment le salaire, l'âge, l'ancienneté, les heures supplémentaires, les déplacements et certains postes plus exposés.

La modélisation confirme qu'il est possible d'entraîner un modèle de classification pour détecter les profils les plus à risque.  
Le modèle Dummy sert de référence minimale, puis la régression logistique et le Random Forest permettent d'améliorer la détection.  
Le Random Forest optimisé est retenu comme modèle final, car il est non linéaire, adapté aux interactions entre variables et compatible avec l'interprétation SHAP.

L'utilisation de SHAP permet de passer d'un modèle prédictif à une lecture métier : on peut identifier les variables qui influencent globalement le risque d'attrition et expliquer localement certains cas salariés.

**Recommandations RH possibles :**

- surveiller les salariés soumis aux heures supplémentaires ;
- analyser les postes et départements les plus exposés ;
- travailler sur les politiques de rémunération ;
- réduire les déplacements lorsque cela est possible ;
- renforcer l'accompagnement des salariés jeunes ou avec peu d'ancienneté ;
- suivre régulièrement les scores de satisfaction et d'équilibre vie professionnelle / vie personnelle.

# Script oral rapide pour présenter cette partie

> Dans cette partie, nous avons finalisé la mission en construisant des modèles de classification.  
> Nous avons d'abord enrichi les données avec quelques variables métier, par exemple une satisfaction moyenne, un indicateur de charge opérationnelle ou encore un indicateur de promotion ancienne.  
> Ensuite, nous avons séparé les données en jeu d'apprentissage et jeu de test avec une stratification, car les salariés ayant quitté l'entreprise sont minoritaires.  
> Nous avons entraîné trois modèles : un Dummy, qui sert de référence minimale, une régression logistique, puis un Random Forest.  
> Pour comparer les modèles, nous avons utilisé plusieurs métriques : accuracy, precision, recall, f1-score, ROC-AUC et average precision.  
> Dans ce contexte RH, le recall est important, car il mesure la capacité du modèle à détecter les salariés qui vont réellement partir.  
> Nous avons aussi utilisé la validation croisée pour vérifier que le modèle est stable et qu'il ne dépend pas uniquement d'un seul découpage train/test.  
> Ensuite, nous avons optimisé le Random Forest avec une recherche d'hyperparamètres.  
> Enfin, nous avons utilisé les méthodes d'interprétation, notamment la permutation importance et SHAP, pour comprendre quelles variables influencent le plus les prédictions.  
> L'intérêt de SHAP est qu'il permet d'expliquer le modèle globalement, mais aussi salarié par salarié avec des graphiques locaux.  
> Au final, le modèle ne sert pas uniquement à prédire : il sert surtout à donner aux RH des pistes concrètes pour réduire l'attrition.